# Transformer Training Notebook for German to DGS Translation

This notebook demonstrates the training of a Transformer model for translating German text into German Sign Language (DGS) tokens. We use:

1. A pretrained German tokenizer (from Hugging Face) to tokenize the source sentences.
2. A custom vocabulary for the DGS tokens.
3. A custom `DataLoader` with translation data from Universität Hamburg's DGS Korpus project.
4. A Transformer model with encoder and decoder.
5. Standard training and evaluation loops.

## Setup and Imports

In [6]:

import math
import copy
import random
import ast
import re
import os
import numpy as np
import pandas as pd
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from torch.utils.tensorboard import SummaryWriter

from transformer import Tokenizer, clean_token

import warnings
from torch.jit import TracerWarning
warnings.filterwarnings("ignore", category=TracerWarning)

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
   torch.cuda.manual_seed(SEED)
   print("cuda")

cuda


## Data Loading and Preprocessing

1. Load dataset (`dataset.csv`)
2. Load pretrained German tokenizer (`dbmdz/bert-base-german-cased`) to tokenize the German sentences.
3. Simplify DGS tokens.

The data is split into training and validation sets.

In [7]:
# from google.colab import drive
# drive.mount('/content/drive')

In [8]:
# Load the dataset
file_path = "dataset.csv"
df = pd.read_csv(file_path)
print("Dataset shape:", df.shape)
print(df.head())

# Convert the DGS column from a string representation to a Python list
df['dgs'] = df['dgs'].apply(lambda x: ast.literal_eval(x))

# You can inspect a sample:
print("\nSample DGS tokens:", df['dgs'].iloc[0])

Dataset shape: (64746, 5)
  transcript  index                                                 de  \
0    1176340      0  Sieh her, jetzt musst du meine Geschichte aus ...   
1    1176340      1  Was ich dir von mir schon erzählt habe, kannst...   
2    1176340      2  Dann kannst du sagen, dass das etwas Neues für...   
3    1176340      3  Ich beginne das Thema von Anfang an, so wie bi...   
4    1176340      4       Mein Thema ist meine Firma, bei der ich bin.   

                                                 dgs  \
0  ['SEHEN-AUF1^*', 'MUSS1*', 'HIRN1A^', 'LÖSCHEN...   
1  ['MEIN1*', 'VERGANGENHEIT1^*', '$INDEX1', 'HIR...   
2     ['$INDEX1*', 'WISSEN2B^', 'STIMMT1B', 'NEU1A']   
3  ['ÜBER1', 'THEMA1*', 'ANFANG1A*', 'WACHSEN2A^*...   
4  ['ICH1', 'THEMA1*', 'ZUSAMMENHANG1A^', 'FIRMA1...   

                                          mouth  
0    ['muss', 'löschen', 'meine', 'geschichte']  
1        ['mei{ne}', 'löschen', 'wiederholung']  
2                     ['[MG]', 'stimmt', '

## Build Vocabulary for DGS

Build a vocabulary from all DGS tokens in the dataset with added special tokens:

- `<pad>`: Padding token
- `<sos>`: Start-of-sequence
- `<eos>`: End-of-sequence
- `<name>`: Name token
- `<num>`: Number token
- `<unk>`: Unknown token

In [9]:

# Special tokens
PAD_TOKEN = "<pad>"
SOS_TOKEN = "<sos>"
EOS_TOKEN = "<eos>"
NAME_TOKEN = "<name>"
NUM_TOKEN = "<num>"
UNK_TOKEN = "<unk>"
max_tokens = 10

special_tokens = [PAD_TOKEN, SOS_TOKEN, EOS_TOKEN, NAME_TOKEN, NUM_TOKEN, UNK_TOKEN]

def build_vocab(token_lists, min_freq=1):
    counter = Counter()
    for tokens in token_lists:
        counter.update(tokens)
    # Get the set of tokens that occur at least min_freq times
    vocab = {tok for tok, cnt in counter.items() if cnt >= min_freq}
    # Remove any tokens that are already in our special token list
    vocab = vocab - set(special_tokens)
    # Sort the remaining tokens
    vocab = sorted(list(vocab))
    # Prepend the special tokens (in the order you want them)
    vocab = special_tokens + vocab
    token2idx = {token: idx for idx, token in enumerate(vocab)}
    idx2token = {idx: token for token, idx in token2idx.items()}
    return token2idx, idx2token

# Build vocabulary from the DGS column
df['dgs'] = df['dgs'].apply(lambda tokens: [clean_token(token) for token in tokens])
dgs_token2idx, dgs_idx2token = build_vocab(df['dgs'].tolist())
df = df[~df['dgs'].apply(lambda tokens: any(("$LIST" in token or "$ALPHA" in token) for token in tokens))].reset_index(drop=True)
df = df[df['dgs'].apply(lambda tokens: len(tokens) <= max_tokens)].reset_index(drop=True)
print("DGS vocab size:", len(dgs_token2idx))
print("Elements in dataset:", len(df["transcript"]))

DGS vocab size: 4685
Elements in dataset: 58663


In [10]:
import json
dgs_vocab = {"token2idx": dgs_token2idx, "idx2token": dgs_idx2token}
with open("dgs_vocab.json", 'w') as f:
    json.dump(dgs_vocab, f)

## Pretrained German Tokenizer

Pretrained German tokenizer from Hugging Face: `dbmdz/bert-base-german-cased`

In [11]:
de_tokenizer = Tokenizer()
print("German tokenizer vocab size:", de_tokenizer.vocab_size)

German tokenizer vocab size: 31105


## Custom Dataset and DataLoader

The `DataLoader` returns:
- **Source:** Tokenized German sentence (list of token ids).
- **Target:** DGS token indices (with `<sos>` prepended and `<eos>` appended).

We also define a collate function that pads sequences to the maximum length in a batch.

In [12]:
class TranslationDataset(Dataset):
    def __init__(self, df, de_tokenizer, dgs_token2idx, max_src_len=32, max_tgt_len=32):
        self.df = df.reset_index(drop=True)
        self.de_tokenizer = de_tokenizer
        self.dgs_token2idx = dgs_token2idx
        self.max_src_len = max_src_len
        self.max_tgt_len = max_tgt_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # Get German sentence and tokenize using the pretrained tokenizer
        src_text = self.df.loc[idx, "de"]
        src_tokens = self.de_tokenizer.encode(src_text, max_length=self.max_src_len)

        # Get DGS tokens and add <sos> and <eos>
        tgt_tokens = self.df.loc[idx, "dgs"]
        tgt_tokens = [SOS_TOKEN] + tgt_tokens + [EOS_TOKEN]
        # Convert to indices using our DGS vocabulary
        tgt_indices = [self.dgs_token2idx.get(tok, self.dgs_token2idx[UNK_TOKEN]) for tok in tgt_tokens]
        # Truncate target if needed
        tgt_indices = tgt_indices[:self.max_tgt_len]

        sample = {
            "src": torch.tensor(src_tokens, dtype=torch.long),
            "tgt": torch.tensor(tgt_indices, dtype=torch.long)
        }
        return sample

def collate_fn(batch):
    # Batch is a list of samples. We pad the src and tgt sequences separately.
    src_seqs = [b["src"] for b in batch]
    tgt_seqs = [b["tgt"] for b in batch]

    src_lengths = [len(seq) for seq in src_seqs]
    tgt_lengths = [len(seq) for seq in tgt_seqs]

    src_padded = nn.utils.rnn.pad_sequence(src_seqs, padding_value=de_tokenizer.pad_token_id, batch_first=True)
    tgt_padded = nn.utils.rnn.pad_sequence(tgt_seqs, padding_value=dgs_token2idx[PAD_TOKEN], batch_first=True)

    return {"src": src_padded, "tgt": tgt_padded, "src_lengths": src_lengths, "tgt_lengths": tgt_lengths}

## Split Data and Create DataLoaders

In [13]:
train_df, val_df = train_test_split(df, test_size=0.05, random_state=SEED)
train_dataset = TranslationDataset(train_df, de_tokenizer, dgs_token2idx)
val_dataset = TranslationDataset(val_df, de_tokenizer, dgs_token2idx)

BATCH_SIZE = 256
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

## Transformer Model Definition

- The encoder embeds German token ids.
- The decoder works with DGS vocabulary indices.

The model accepts source and target sequences with expected shape: `[seq_len, batch_size]`)

In [14]:
# Model definition in transformer.py

from transformer import Transformer

## Training and Evaluation Functions

- `create_masks`: to create subsequent masks for the decoder.
- `train_epoch`: a function that runs one training epoch.
- `evaluate`: meassure model performance on the validation set.

The model expects input shapes of (seq_len, batch_size) so batch data is transposed.

Teacher forcing: decoder input is the target sequence without the last token; target for loss is the target sequence without the first token.

In [15]:

def create_masks(src, tgt, src_pad_idx, tgt_pad_idx):
    """
    Create key padding masks for src and tgt.
    Note: For the target, the full sequence mask is computed first.
    """
    src_key_padding_mask = (src == src_pad_idx)
    tgt_key_padding_mask = (tgt == tgt_pad_idx)
    return src_key_padding_mask, tgt_key_padding_mask

def train_epoch(model, dataloader, optimizer, criterion, src_pad_idx, tgt_pad_idx, device, writer, epoch):
    model.train()
    epoch_loss = 0
    for i, batch in enumerate(dataloader):
        src = batch["src"].to(device)  # (batch_size, src_seq_len)
        tgt = batch["tgt"].to(device)  # (batch_size, tgt_seq_len)
        optimizer.zero_grad()

        # Transpose for the model: (seq_len, batch_size)
        src = src.transpose(0, 1)
        tgt = tgt.transpose(0, 1)

        # Teacher forcing:
        #   decoder input: all but the last token (shape: (tgt_seq_len-1, batch_size))
        #   target output: all but the first token (flattened)
        tgt_input = tgt[:-1, :]
        tgt_out = tgt[1:, :].contiguous().view(-1)

        max_idx = tgt_out.max().item()
        if max_idx >= len(dgs_token2idx):
            print("Error: found target index", max_idx, "but TGT_VOCAB_SIZE =", len(dgs_token2idx))

        # Create key padding masks (computed on the full sequence)
        src_kpm, tgt_kpm_full = create_masks(src.transpose(0,1), tgt.transpose(0,1), src_pad_idx, tgt_pad_idx)
        # Slice target key padding mask to match the decoder input length (tgt[:-1])
        tgt_kpm = tgt_kpm_full[:, :-1]

        # Create the target mask based on the decoder input length
        tgt_seq_len = tgt_input.size(0)
        tgt_mask = torch.triu(torch.ones((tgt_seq_len, tgt_seq_len), device=src.device) == 1).transpose(0, 1)
        tgt_mask = tgt_mask.float().masked_fill(tgt_mask == 0, float('-inf')).masked_fill(tgt_mask == 1, float(0.0))

        # if i < 2:
        #     print("\nModel Input:")
        #     print("Source Tensor:", src.shape)
        #     print("Target Tensor:", tgt_input.shape)
        #     print("Target Mask:", tgt_mask.shape)
        #     print("Source Padding Mask:", src_kpm.shape)
        #     print("Target Padding Mask:", tgt_kpm.shape, "\n")

        output = model(src, tgt_input, src_mask=None, tgt_mask=tgt_mask,
                       src_key_padding_mask=src_kpm,
                       tgt_key_padding_mask=tgt_kpm,
                       memory_key_padding_mask=src_kpm)
        # output: (tgt_seq_len, batch_size, tgt_vocab_size)
        output = output.view(-1, output.shape[-1])

        loss = criterion(output, tgt_out)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        if i % 1 == 0:
            writer.add_scalar("Loss/train_batch", loss.item(), epoch * len(dataloader) + i)
    return epoch_loss / len(dataloader)

def evaluate(model, dataloader, criterion, src_pad_idx, tgt_pad_idx, device):
    model.eval()
    epoch_loss = 0
    with torch.no_grad():
        for batch in dataloader:
            src = batch["src"].to(device)
            tgt = batch["tgt"].to(device)

            src = src.transpose(0, 1)
            tgt = tgt.transpose(0, 1)

            tgt_input = tgt[:-1, :]
            tgt_out = tgt[1:, :].contiguous().view(-1)

            src_kpm, tgt_kpm_full = create_masks(src.transpose(0,1), tgt.transpose(0,1), src_pad_idx, tgt_pad_idx)
            tgt_kpm = tgt_kpm_full[:, :-1]

            tgt_seq_len = tgt_input.size(0)
            tgt_mask = torch.triu(torch.ones((tgt_seq_len, tgt_seq_len), device=src.device) == 1).transpose(0, 1)
            tgt_mask = tgt_mask.float().masked_fill(tgt_mask == 0, float('-inf')).masked_fill(tgt_mask == 1, float(0.0))

            output = model(src, tgt_input, src_mask=None, tgt_mask=tgt_mask,
                           src_key_padding_mask=src_kpm,
                           tgt_key_padding_mask=tgt_kpm,
                           memory_key_padding_mask=src_kpm)
            output = output.view(-1, output.shape[-1])
            loss = criterion(output, tgt_out)
            epoch_loss += loss.item()
    return epoch_loss / len(dataloader)

## Training Loop

In [ ]:

# Model Configurations: 1       2       3       4       5       6
# config = {"d":      [   128,    256,    256,    512 ],
#           "el":     [   3,      3,      3,      3   ],
#           "dl":     [   3,      3,      3,      3   ],
#           "ff":     [   256,    256,    512,    512 ],
#           "ep":     [   10,     30,     30,     30  ],
#           "prevep": [   20,     0,      0,      0   ]}

config = {"d":      [   32  ],
          "el":     [   1   ],
          "dl":     [   1   ],
          "ff":     [   32  ],
          "ep":     [   30  ],
          "prevep": [   30   ]}

log_dir = "eval/progress"
state_dir = "eval/state"
os.makedirs(log_dir, exist_ok=True)
os.makedirs(state_dir, exist_ok=True)

for d, el, dl, ff, ep, prevep in zip(config["d"], config["el"], config["dl"], config["ff"], config["ep"], config["prevep"]):

    # Device configuration
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)

    # Model hyperparameters
    MODEL_DIM = d
    NUM_HEADS = 8
    NUM_ENCODER_LAYERS = el
    NUM_DECODER_LAYERS = dl
    FF_DIM = ff
    DROPOUT = 0.2

    model_id = f"d{MODEL_DIM}_el{NUM_ENCODER_LAYERS}_dl{NUM_DECODER_LAYERS}_ff{FF_DIM}"
    print(model_id)
    
    log_file = os.path.join(log_dir, f"{model_id}.csv")
    state_file = os.path.join(state_dir, f"{model_id}.pt")
    state_file_minval = os.path.join(state_dir, f"{model_id}_minval.pt")

    if not os.path.exists(log_file):
        with open(log_file, "w") as f:
            f.write("Epoch,Training Loss,Validation Loss\n")

    # DE -> tokenizer vocab, DGS -> custom vocab
    SRC_VOCAB_SIZE = de_tokenizer.vocab_size
    TGT_VOCAB_SIZE = len(dgs_token2idx)

    model = Transformer(SRC_VOCAB_SIZE, TGT_VOCAB_SIZE, model_dim=MODEL_DIM, num_heads=NUM_HEADS,
                        num_encoder_layers=NUM_ENCODER_LAYERS, num_decoder_layers=NUM_DECODER_LAYERS,
                        ff_dim=FF_DIM, dropout=DROPOUT).to(device)

    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total learnable parameters: {total_params}")

    optimizer = optim.Adam(model.parameters(), lr=0.002)
    criterion = nn.CrossEntropyLoss(ignore_index=dgs_token2idx[PAD_TOKEN])

    # TensorBoard writer
    writer = SummaryWriter(log_dir="runs/"+model_id)
    # Log model graph with a sample input
    sample_src = torch.randint(0, SRC_VOCAB_SIZE, (35, 1), dtype=torch.long, device=device)
    sample_tgt = torch.randint(0, TGT_VOCAB_SIZE, (20, 1), dtype=torch.long, device=device)
    # Dummy masks
    dummy_src_mask = torch.zeros((sample_src.size(0), sample_src.size(0)), device=device)  # shape: (35, 35)
    dummy_tgt_mask = torch.zeros((sample_tgt.size(0), sample_tgt.size(0)), device=device)  # shape: (20, 20)
    dummy_memory_mask = torch.zeros((sample_tgt.size(0), sample_src.size(0)), device=device)  # shape: (20, 35)
    # No padding as key padding
    dummy_src_key_padding_mask = torch.zeros((sample_src.size(1), sample_src.size(0)), dtype=torch.bool, device=device)  # shape: (1, 35)
    dummy_tgt_key_padding_mask = torch.zeros((sample_tgt.size(1), sample_tgt.size(0)), dtype=torch.bool, device=device)  # shape: (1, 20)
    dummy_memory_key_padding_mask = torch.zeros((sample_src.size(1), sample_src.size(0)), dtype=torch.bool, device=device)  # shape: (1, 35)

    writer.add_graph(model, (sample_src, sample_tgt, dummy_src_mask, dummy_tgt_mask, dummy_memory_mask, dummy_src_key_padding_mask, 
                            dummy_tgt_key_padding_mask, dummy_memory_key_padding_mask))

    try:
        model.load_state_dict(torch.load(state_file, weights_only=True)) ###
    except Exception as e:
        print("No saved model states found; training from scratch.")

    min_val_loss = 1000

    NUM_EPOCHS = ep
    prev_epochs = prevep

    for epoch in range(1, NUM_EPOCHS+1):
        total_epoch = prev_epochs + epoch
        train_loss = train_epoch(model, train_loader, optimizer, criterion,
                                src_pad_idx=de_tokenizer.pad_token_id,
                                tgt_pad_idx=dgs_token2idx[PAD_TOKEN], device=device, 
                                writer=writer, epoch=total_epoch)
        val_loss = evaluate(model, val_loader, criterion,
                            src_pad_idx=de_tokenizer.pad_token_id,
                            tgt_pad_idx=dgs_token2idx[PAD_TOKEN], device=device)
        
        writer.add_scalar("Loss/train_epoch", train_loss, total_epoch)
        writer.add_scalar("Loss/val_epoch", val_loss, total_epoch)
        print(f"Epoch {total_epoch}: Train Loss = {train_loss:.4f} | Val Loss = {val_loss:.4f}")
        writer.add_scalar("LearningRate", optimizer.param_groups[0]["lr"], total_epoch)
        torch.save(model.state_dict(), state_file)

        if val_loss < min_val_loss:
            torch.save(model.state_dict(), state_file_minval)
            min_val_loss = val_loss

        with open(log_file, "a") as f:
            f.write(f"{total_epoch},{train_loss},{val_loss}\n")

    writer.close()

Using device: cuda
d32_el1_dl1_ff32
Total learnable parameters: 1317101
Epoch 31: Train Loss = 4.4437 | Val Loss = 4.5398
Epoch 32: Train Loss = 4.4178 | Val Loss = 4.5316
Epoch 33: Train Loss = 4.3982 | Val Loss = 4.5220
Epoch 34: Train Loss = 4.3833 | Val Loss = 4.5136
Epoch 35: Train Loss = 4.3671 | Val Loss = 4.5118
Epoch 36: Train Loss = 4.3559 | Val Loss = 4.5037
Epoch 37: Train Loss = 4.3428 | Val Loss = 4.4978
Epoch 38: Train Loss = 4.3249 | Val Loss = 4.4909
Epoch 39: Train Loss = 4.3159 | Val Loss = 4.4840
Epoch 40: Train Loss = 4.3025 | Val Loss = 4.4851
Epoch 41: Train Loss = 4.2897 | Val Loss = 4.4756
Epoch 42: Train Loss = 4.2797 | Val Loss = 4.4764
Epoch 43: Train Loss = 4.2663 | Val Loss = 4.4695
Epoch 44: Train Loss = 4.2562 | Val Loss = 4.4633
Epoch 45: Train Loss = 4.2476 | Val Loss = 4.4589
Epoch 46: Train Loss = 4.2344 | Val Loss = 4.4485
Epoch 47: Train Loss = 4.2243 | Val Loss = 4.4475
Epoch 48: Train Loss = 4.2166 | Val Loss = 4.4468
Epoch 49: Train Loss = 4.208

## Inference Function

In [17]:

def greedy_decode(model, src_sentence, max_len, de_tokenizer, dgs_token2idx, dgs_idx2token, device):
    model.eval()
    # Tokenize source sentence
    src_tokens = de_tokenizer.encode(src_sentence)
    src_tensor = torch.tensor(src_tokens, dtype=torch.long).unsqueeze(0).to(device)  # (1, src_seq_len)
    src_tensor = src_tensor.transpose(0,1)  # (src_seq_len, 1)

    # No masks for source here (or create them as needed)
    memory = model.encoder(model.pos_encoder(model.src_embedding(src_tensor) * math.sqrt(model.model_dim)))

    # Initialize target sequence with <sos>
    tgt_indices = [dgs_token2idx[SOS_TOKEN]]
    for _ in range(max_len):
        tgt_tensor = torch.tensor(tgt_indices, dtype=torch.long).unsqueeze(1).to(device)  # (tgt_seq_len, 1)
        tgt_tensor = model.pos_decoder(model.tgt_embedding(tgt_tensor) * math.sqrt(model.model_dim))
        tgt_mask = torch.triu(torch.ones((tgt_tensor.size(0), tgt_tensor.size(0)), device=device) == 1).transpose(0, 1)
        tgt_mask = tgt_mask.float().masked_fill(tgt_mask == 0, float('-inf')).masked_fill(tgt_mask == 1, float(0.0))

        out = model.decoder(tgt_tensor, memory, tgt_mask=tgt_mask)
        out = model.fc_out(out)
        # Get the last token's prediction
        prob = out[-1, 0]
        next_token = torch.argmax(prob).item()
        tgt_indices.append(next_token)
        if next_token == dgs_token2idx[EOS_TOKEN]:
            break

    # Convert indices to tokens (excluding <sos> and <eos>)
    decoded_tokens = [dgs_idx2token[idx] for idx in tgt_indices if idx not in {dgs_token2idx[SOS_TOKEN], dgs_token2idx[EOS_TOKEN]}]
    return decoded_tokens

## Testing Inference

In [18]:
example_sentence = "Morgen ist Dienstag."
versions = ["", "_minval"]

for d, el, dl, ff in zip(config["d"], config["el"], config["dl"], config["ff"]):
    model_id = f"d{d}_el{el}_dl{dl}_ff{ff}"
    model = Transformer(SRC_VOCAB_SIZE, TGT_VOCAB_SIZE, model_dim=d, num_heads=8,
                        num_encoder_layers=el, num_decoder_layers=dl,
                        ff_dim=ff, dropout=0.2).to(device)
    
    for version in versions:
        state_file = os.path.join(state_dir, model_id + version + ".pt")
        
        try:
            model.load_state_dict(torch.load(state_file, weights_only=True))
        except Exception as e:
            print("No saved state found. Skipping inference for version", state_file)

        decoded_dgs = greedy_decode(model, example_sentence, max_len=20,
                                    de_tokenizer=de_tokenizer,
                                    dgs_token2idx=dgs_token2idx, dgs_idx2token=dgs_idx2token, device=device)
        print(model_id, version)
        print(" Input German sentence:", example_sentence)
        print(" Decoded DGS tokens:", decoded_dgs)

d32_el1_dl1_ff32 
 Input German sentence: Morgen ist Dienstag.
 Decoded DGS tokens: ['index', 'viel', 'leben']
d32_el1_dl1_ff32 _minval
 Input German sentence: Morgen ist Dienstag.
 Decoded DGS tokens: ['ich', 'index', 'ich', 'index']
